In [7]:
import pandas as pd

df=pd.read_csv('../datasets/raw/cfpb_credit_card_complaints.csv')

# print(df.head()) 
print(df.shape)
print('\n\n')
print(df.info())

(83590, 16)



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 83590 entries, 0 to 83589
Data columns (total 16 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   Date received                 83590 non-null  object
 1   Product                       83590 non-null  object
 2   Sub-product                   76492 non-null  object
 3   Issue                         83590 non-null  object
 4   Sub-issue                     76327 non-null  object
 5   Consumer complaint narrative  83590 non-null  object
 6   Company public response       40022 non-null  object
 7   Company                       83590 non-null  object
 8   State                         83284 non-null  object
 9   ZIP code                      83583 non-null  object
 10  Tags                          17020 non-null  object
 11  Submitted via                 83590 non-null  object
 12  Date sent to company          83590 non-null  object
 13  C

In [8]:
required_columns = [
    "Complaint ID",
    "Date received",
    "Product",
    "Sub-product",
    "Issue",
    "Sub-issue",
    "Consumer complaint narrative",
    "Company",
    "Company response to consumer",
    "Timely response?"
]

df = df[required_columns]

In [9]:
df.head()

,Complaint ID,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company,Company response to consumer,Timely response?
0,7634786,2023-10-03T14:48:38.000Z,Credit card,General-purpose credit card or charge card,Problem with a company's investigation into an...,Was not notified of investigation status or re...,I'm having difficulty accepting this issue and...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",Closed with non-monetary relief,Yes
1,6681064,2023-03-11T15:29:34.000Z,Credit card or prepaid card,General-purpose credit card or charge card,Problem with a purchase shown on your statement,Card was charged for something you did not pur...,I called Discover to report my card lost. I to...,DISCOVER BANK,Closed with explanation,Yes
2,6681463,2023-03-11T15:35:55.000Z,Credit card or prepaid card,General-purpose credit card or charge card,Problem with a purchase shown on your statement,Card was charged for something you did not pur...,My TD Bank Business Credit Card XXXX was last ...,TD BANK US HOLDING COMPANY,Closed with explanation,Yes
3,6681185,2023-03-11T15:59:56.000Z,Credit card or prepaid card,General-purpose credit card or charge card,Problem with a purchase shown on your statement,Credit card company isn't resolving a dispute ...,On XX/XX/XXXX I ordered a TV online from a ret...,JPMORGAN CHASE & CO.,Closed with explanation,Yes
4,7634456,2023-10-03T15:30:39.000Z,Credit card,General-purpose credit card or charge card,Problem with a company's investigation into an...,Was not notified of investigation status or re...,It is completely unjustified that I have consi...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",Closed with non-monetary relief,Yes


In [10]:
df = df.rename(columns={
    "Complaint ID": "complaint_id",
    "Date received": "date_received",
    "Product": "product",
    "Sub-product": "sub_product",
    "Issue": "issue",
    "Sub-issue": "sub_issue",
    "Consumer complaint narrative": "complaint_text",
    "Company": "company",
    "Company response to consumer": "company_response",
    "Timely response?": "timely_response"
})

In [11]:
df.duplicated().sum()

np.int64(0)

In [12]:
df["complaint_text"] = (
    df["complaint_text"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [13]:
df["product"].value_counts()

product
Credit card                    46995
Credit card or prepaid card    36595
Name: count, dtype: int64

In [14]:
df["sub_product"].value_counts()

sub_product
General-purpose credit card or charge card    63530
Store credit card                              7292
General-purpose prepaid card                   2919
Government benefit card                        2001
Gift card                                       470
Payroll card                                    269
Student prepaid card                             11
Name: count, dtype: int64

In [15]:
df["issue"].value_counts()

issue
Problem with a purchase shown on your statement                    52452
Problem with a company's investigation into an existing problem     8933
Trouble using your card                                             8861
Problem with a purchase or transfer                                 3116
Billing disputes                                                    3102
Trouble using the card                                              2554
Identity theft / Fraud / Embezzlement                               1722
Customer service / Customer relations                                973
Transaction issue                                                    681
Billing statement                                                    620
Credit monitoring or identity theft protection services              411
Problem with fraud alerts or security freezes                        165
Name: count, dtype: int64

In [16]:
df["sub_issue"].value_counts()

sub_issue
Credit card company isn't resolving a dispute about a purchase on your statement         36962
Card was charged for something you did not purchase with the card                        13916
Can't use card to make purchases                                                          6000
Was not notified of investigation status or results                                       5747
Credit card company won't increase or decrease your credit limit                          2726
Their investigation did not fix an error on your report                                   2529
Card company isn't resolving a dispute about a purchase or transfer                       1885
Overcharged for something you did purchase with the card                                  1574
Trouble using the card to spend money in a store or online                                1288
Charged for a purchase or transfer you did not make with the card                         1174
Trouble getting information about the ca

In [17]:
df["company_response"].value_counts()

company_response
Closed with explanation            55434
Closed with monetary relief        18703
Closed with non-monetary relief     9347
Untimely response                     63
Closed                                42
In progress                            1
Name: count, dtype: int64

In [18]:
sample = df.sample(30, random_state=42)

for _, row in sample.iterrows():
    print("=" * 100)
    print("Issue:", row["issue"])
    print("Sub-Issue:", row["sub_issue"])
    print(row["complaint_text"][:1000])

Issue: Problem with a purchase shown on your statement
Sub-Issue: Card was charged for something you did not purchase with the card
There was fraudulent charge made on my card at a wireless store. I notified Barclay by phone 5 times emailed and faxed them over a 2 month period. I have been unknowingly paying for the charge for over a year because I have all cards set up on autopay and recently relized when my bank account was compromised and had to set up all accounts again on new account and realized I had two Barclays payments coming out. I am getting the run around and no help from Barclays!
Issue: Problem with a company's investigation into an existing problem
Sub-Issue: Was not notified of investigation status or results
My accounts was never late! I have had exceptional payment history with XXXX XXXX and all payments were placed on XXXX. This late payment that is reporting is a result on a systematic error on their end processing my payment. This was clearly a billing error made 

In [19]:
# Drop rows with no complaint text (they can't be labeled or trained on)
print(f"Rows before dropping nulls: {len(df):,}")
df = df.dropna(subset=['complaint_text'])
print(f"Rows after dropping null complaint_text: {len(df):,}")

import re

def clean_text(text):
    text = str(text).lower()
    # Remove CFPB redaction placeholders (XXXX, XX/XX/XXXX etc.)
    text = re.sub(r'\bxx+\b', '', text)
    text = re.sub(r'\b\d{1,2}/\d{1,2}/\d{4}\b', '', text)  # dates
    # Remove dollar amounts like $1,200.00
    text = re.sub(r'\$[\d,]+\.?\d*', 'AMOUNT', text)
    # Remove URLs and emails
    text = re.sub(r'http\S+|www\S+|\S+@\S+', '', text)
    # Remove non-alphabetic characters (keep spaces)
    text = re.sub(r'[^a-z\s]', ' ', text)
    # Collapse multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['complaint_text_clean'] = df['complaint_text'].apply(clean_text)

# Filter out complaints that are too short after cleaning
df['word_count'] = df['complaint_text_clean'].str.split().str.len()
df = df[df['word_count'] >= 10]

print(f"Final rows after cleaning: {len(df):,}")
print(f"Avg complaint length: {df['word_count'].mean():.0f} words")

Rows before dropping nulls: 83,590
Rows after dropping null complaint_text: 83,590
Final rows after cleaning: 83,349
Avg complaint length: 226 words


In [21]:
# Overwrite the clean_complaints.csv with the improved version
df.to_csv("../datasets/processed/clean_complaints.csv", index=False)
print(f"✅ Saved {len(df):,} rows to clean_complaints.csv")
print(f"   Columns: {df.columns.tolist()}")

✅ Saved 83,349 rows to clean_complaints.csv
   Columns: ['complaint_id', 'date_received', 'product', 'sub_product', 'issue', 'sub_issue', 'complaint_text', 'company', 'company_response', 'timely_response', 'complaint_text_clean', 'word_count']
